# Results of the completed pipeline run

Displays the selected model type and **2024 test accuracy** for execution `229c5ca9ad2f4bc9b3c7e876815350ab`.

Reads saved outputs from `00_run_pipeline.ipynb` and checks them against `outputs/test_predictions.csv`. Running all cells only reads existing results; it does not run the pipeline or train a model. Test accuracy is distinct from the historical mean accuracy used to select the model. The execution ID is pinned so a later run cannot silently replace these results.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

# Read files only: no pipeline, training, or evaluation imports.
EXPECTED_EXECUTION_ID = "229c5ca9ad2f4bc9b3c7e876815350ab"
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "notebooks" / "00_run_pipeline.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the project directory or notebooks folder.")

saved = json.loads((root / "notebooks" / "00_run_pipeline.ipynb").read_text())
texts = []
for cell in saved["cells"]:
    for output in cell.get("outputs", []):
        value = output.get("text", output.get("data", {}).get("text/plain", ""))
        texts.append("".join(value) if isinstance(value, list) else value)
text = "\n".join(texts)
match = re.search(r"Selected model: (.*?); status: (.*?); execution: ([0-9a-f]+)", text)
if not match:
    raise ValueError("No saved completed-run summary found in 00_run_pipeline.ipynb.")
model_type, status, execution_id = match.groups()
if execution_id != EXPECTED_EXECUTION_ID or status != "complete":
    raise ValueError("The source notebook no longer contains the requested completed run.")
metric_text = next((t for t in texts if re.search(r"^accuracy\s+", t)
                    and "evaluation_rows" in t), None)
if metric_text is None:
    raise ValueError("The saved final test metrics are missing.")
metrics = {m.group(1): float(m.group(2)) for m in
           re.finditer(r"^(\w+)\s+(-?\d+(?:\.\d+)?)\s*$", metric_text, re.MULTILINE)}

# Confirm the saved predictions agree with the recorded test accuracy.
predictions = pd.read_csv(root / "notebooks" / "outputs" / "test_predictions.csv")
if predictions[["winner", "predicted_winner"]].isna().any().any():
    raise ValueError("Saved predictions contain missing winners.")
correct = int(predictions["winner"].eq(predictions["predicted_winner"]).sum())
accuracy = correct / len(predictions)
if len(predictions) != int(metrics["evaluation_rows"]) or abs(accuracy - metrics["accuracy"]) > 0.000001:
    raise ValueError("Saved predictions do not match this run's recorded accuracy/count.")

results = pd.DataFrame([{
    "Selected model type": model_type,
    "2024 test accuracy": f"{accuracy:.2%}",
    "Correct predictions": correct,
    "Evaluated seats": len(predictions),
    "Run status": status,
    "Execution ID": execution_id,
}])
display(results)
results


,Selected model type,2024 test accuracy,Correct predictions,Evaluated seats,Run status,Execution ID
0,Conditional XGBoost,73.73%,466,632,complete,229c5ca9ad2f4bc9b3c7e876815350ab


,Selected model type,2024 test accuracy,Correct predictions,Evaluated seats,Run status,Execution ID
0,Conditional XGBoost,73.73%,466,632,complete,229c5ca9ad2f4bc9b3c7e876815350ab
